Load Python packages

In [1]:
!pip -q install openai faiss-cpu pymupdf tiktoken jsonschema tenacity

In [2]:
import os, json, re, math, time, uuid, textwrap
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
import numpy as np
import faiss
import fitz  # PyMuPDF
from openai import OpenAI
from jsonschema import validate, Draft202012Validator
from tenacity import retry, stop_after_attempt, wait_exponential
import tiktoken

RUN_ID = time.strftime('%Y%m%d-%H%M%S')
os.makedirs('outputs', exist_ok=True)
print('OK: imports ready')

OK: imports ready


Login to OpenAI

In [3]:
import os, getpass
from openai import OpenAI
if not os.environ.get('OPENAI_API_KEY'):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter OPENAI_API_KEY: ")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

Set up PDF processing utils
- Here we use `PyMuPDF` to process pdf

In [4]:
def read_pdf(path: str) -> List[Dict[str, Any]]:
    """Return list of {page:int, text:str} dicts (1-based pages)."""
    doc = fitz.open(path) # Use fitz (i.e. PyMuPDF) to process pdf
    out = []
    for i, page in enumerate(doc):
        txt = page.get_text('text') or '' # Extract text of each page
        out.append({'page': i+1, 'text': txt}) # Output be like: page + text
    return out

def split_into_chunks(pages: List[Dict[str, Any]], max_chars: int=2000, overlap: int=200) -> List[Dict[str, Any]]:
    """Naive chunker by characters, page-aware; returns chunks with page span.
    Each chunk dict: {id, text, meta:{pages:[int,...], source}}"""
    chunks = []
    buf, buf_pages = '', []
    for p in pages:
        text = p['text']
        ptr = 0
        while ptr < len(text):
            remain = max_chars - len(buf)
            take = text[ptr: ptr+remain]
            buf += take
            if p['page'] not in buf_pages:
                buf_pages.append(p['page'])
            ptr += len(take)
            if len(buf) >= max_chars or ptr >= len(text):
                cid = str(uuid.uuid4())
                chunks.append({
                    'id': cid,
                    'text': buf.strip(),
                    'meta': {'pages': buf_pages.copy()}
                })
                # overlap handling
                if overlap > 0:
                    buf = buf[-overlap:]
                    buf_pages = buf_pages[-1:]
                else:
                    buf, buf_pages = '', []
    return chunks

print('OK: PDF utils ready')

OK: PDF utils ready


Set up *embedding model*.

In [5]:
EMBED_MODEL = 'text-embedding-3-small'  # switch to -large for better performance

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=1, max=10))
def embed_texts(texts: List[str]) -> np.ndarray:
    # OpenAI embeddings: batch up to reasonable size
    res = client.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = [d.embedding for d in res.data]
    return np.array(vecs, dtype='float32')

class SimpleFaiss:
    """A simple wrapper around FAISS for similarity search."""

    def __init__(self):
        self.index = None
        self.meta: List[Dict[str, Any]] = []
        self.dim = None

    def build(self, chunks: List[Dict[str, Any]]):
        """Builds the FAISS index from a list of text chunks."""
        texts = [c['text'] for c in chunks]
        vecs = embed_texts(texts)
        self.dim = vecs.shape[1]
        # normalize for inner product cosine similarity
        faiss.normalize_L2(vecs)
        self.index = faiss.IndexFlatIP(self.dim)
        self.index.add(vecs)
        self.meta = chunks
        return self

    def query(self, q: str, k: int=5) -> List[Dict[str, Any]]:
        """Queries the FAISS index with a given text and returns the top k hits."""
        qv = embed_texts([q]).astype('float32')
        faiss.normalize_L2(qv)
        D, I = self.index.search(qv, k)
        hits = []
        for score, idx in zip(D[0], I[0]):
            if idx == -1: continue
            item = self.meta[idx]
            hits.append({
                'text': item['text'],
                'pages': item['meta'].get('pages', []),
                'score': float(score)
            })
        return hits

print('OK: Embedding + FAISS ready')

OK: Embedding + FAISS ready


Set up prompt and schema

In [ ]:
OUTPUT_SCHEMA = {
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "type": "object",
  "properties": {
    "doc_id": {"type": "string"},
    "primary_type": {"type": "string", "enum": ["T1","T2","T3","T4","undetermined"]},
    "secondary_types": {"type": "array", "items": {"type": "string", "enum": ["T1","T2","T3","T4"]}},
    "dimensions": {
      "type": "object",
      "properties": {
        "utility_dominates": {"type": "object", "properties": {"value": {"type": "boolean"}, "evidence_spans": {"type": "array", "items": {"type": "object", "properties": {"quote": {"type": "string"}, "page_or_loc": {"type": "string"}}, "required": ["quote","page_or_loc"]}}}, "required": ["value","evidence_spans"]},
        "on_the_ground": {"type": "object", "properties": {"value": {"type": "boolean"}, "evidence_spans": {"type": "array", "items": {"type": "object", "properties": {"quote": {"type": "string"}, "page_or_loc": {"type": "string"}}, "required": ["quote","page_or_loc"]}}}, "required": ["value","evidence_spans"]}
      },
      "required": ["utility_dominates","on_the_ground"]
    },
    "whack_a_mole_flags": {"type": "array", "items": {"type": "object", "properties": {"quote": {"type": "string"}, "page_or_loc": {"type": "string"}}}},
    "reasoning_brief": {"type": "string"},
    "confidence": {"type": "number"},
    "disagreements": {"type": "array", "items": {"type": "object"}},
    "citations": {"type": "array", "items": {"type": "object", "properties": {"source": {"type": "string"}, "page": {"type": "string"}}}}
  },
  "required": ["doc_id","primary_type","dimensions","reasoning_brief"]
}

Q_SET = [
  ("Q1", "Does the paper's reasoning center on utility maximization/satisficing? Answer true/false and cite evidence."),
  ("Q2", "Is the analysis grounded in a concrete, on-the-ground problem context? Answer true/false and cite evidence."),
  ("Q3", "Given Q1 and Q2, select the primary 4PT Type (T1/T2/T3/T4). Provide a concise justification (≤120 chars)."),
  ("Q4", "Do you detect any 'whack-a-mole' risk or using T1/2/3 to treat a Type 4 situation? Provide 0–2 short evidence spans.")
]

def format_context(hits: List[Dict[str, Any]], title: str) -> str:
    lines = [f"## {title}"]
    for i, h in enumerate(hits, 1):
        pg = ','.join(map(str, h['pages'])) if h['pages'] else 'NA'
        snippet = h['text'].strip().replace('\n', ' ')
        if len(snippet) > 700: snippet = snippet[:700] + '…'
        lines.append(f"[{i}] (p.{pg}) {snippet}")
    return '\n'.join(lines)

def compose_prompt(doc_id: str, codebook_hits: List[Dict[str, Any]], paper_hits: List[Dict[str, Any]]) -> str:
    context_cb = format_context(codebook_hits, '4PT Codebook Context')
    context_doc = format_context(paper_hits, 'Document Evidence')
    
    # 简化的指令，不包含完整 schema
    instructions = f"""
You are an expert policy analyst applying the 4PT framework. Use ONLY the provided Codebook context and Document Evidence to answer Q1–Q4.

Return **valid JSON** with these fields:
- doc_id: "{doc_id}"
- primary_type: one of ["T1","T2","T3","T4","undetermined"]  
- dimensions: object with utility_dominates and on_the_ground, each having value (boolean) and evidence_spans (array of quote/page_or_loc objects)
- reasoning_brief: concise justification (≤120 chars)
- whack_a_mole_flags: array of quote/page_or_loc objects
- secondary_types: array of applicable types
- confidence: number 0-1
- disagreements: array
- citations: array

Answer as pure JSON, no additional text.
    """.strip()

    questions = '\n'.join([f"- {k}: {v}" for k,v in Q_SET])
    prompt = f"""
{instructions}

{context_cb}

{context_doc}

### Questions
{questions}
    """.strip()
    return prompt

print('OK: Schema & prompt builders ready')

OK: Schema & prompt builders ready


Set up working directory and file location.

- For convience, simply adjust the `CODE_MODE` below (whatever it is right now) to `CODE_MODE= "manual_upload"` and then upload the files manually.

- In order to make the code to successfully detect the pdf, please rename the codebook exactly as 4ptCodebook.pdf and the research article as sample_paper.pdf


In [8]:
CODE_MODE = "local"  # "colab", "local", or "manual_upload"

if CODE_MODE == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    DOC_PATH = "/content/drive/MyDrive/AI44PT/data/processed/sample_paper.pdf"
    CODEBOOK_PATH = "/content/drive/MyDrive/AI44PT/data/processed/4ptCodebook.pdf"
elif CODE_MODE == "local":
    import os
    import pathlib
    cwd = os.getcwd()
    cwd_path = pathlib.Path(cwd)
    cwd = str(cwd_path.parent)
    print(f"Current working directory: {cwd}")
    DOC_PATH = os.path.join(cwd, "data/processed/sample_paper.pdf")
    CODEBOOK_PATH = os.path.join(cwd, "data/processed/4ptCodebook.pdf")
elif CODE_MODE == "manual_upload":
    from google.colab import files
    print('Upload your PDFs. Please name the codebook file "4ptCodebook.pdf" and the paper file "sample_paper.pdf".')
    uploaded = files.upload()  # choose files in the UI
    print('Files uploaded:', list(uploaded.keys()))
    CODEBOOK_FILENAME = "4ptCodebook.pdf"
    DOC_FILENAME = "sample_paper.pdf"

    if CODEBOOK_FILENAME in uploaded and DOC_FILENAME in uploaded:
        CODEBOOK_PATH = CODEBOOK_FILENAME
        DOC_PATH = DOC_FILENAME
        print(f"Codebook file: {CODEBOOK_PATH}, Document file: {DOC_PATH}")
    else:
        print(f"Error: Please ensure you upload both '{CODEBOOK_FILENAME}' and '{DOC_FILENAME}'.")
        CODEBOOK_PATH = None
        DOC_PATH = None

Current working directory: /Users/xinby/Library/CloudStorage/GoogleDrive-letheon.by@gmail.com/My Drive/AI44PT


Process specific codebook and research paper using function defined above

In [9]:

cb_pages = read_pdf(CODEBOOK_PATH)
paper_pages = read_pdf(DOC_PATH)
print(f'Codebook pages: {len(cb_pages)}, Paper pages: {len(paper_pages)}')

cb_chunks = split_into_chunks(cb_pages, max_chars=2000, overlap=200)
paper_chunks = split_into_chunks(paper_pages, max_chars=1800, overlap=200)
print(f'Chunks — Codebook: {len(cb_chunks)} / Paper: {len(paper_chunks)}')

cb_index = SimpleFaiss().build(cb_chunks)
paper_index = SimpleFaiss().build(paper_chunks)
print(f'Built FAISS indexes. dims={cb_index.dim}, codebook_vectors={len(cb_index.meta)}, paper_vectors={len(paper_index.meta)}')

Codebook pages: 51, Paper pages: 19
Chunks — Codebook: 130 / Paper: 65
Built FAISS indexes. dims=1536, codebook_vectors=130, paper_vectors=65


In [14]:
LLM_MODEL = 'gpt-4o-mini' # switch to 'gpt-4o' for better performance

def rag_answer(doc_id: str, question: str, top_k_cb: int=5, top_k_doc: int=5, model: 
    str='gpt-4o-mini') -> Dict[str, Any]:
    cb_hits = cb_index.query(question, k=top_k_cb)
    doc_hits = paper_index.query(question, k=top_k_doc)
    prompt = compose_prompt(doc_id, cb_hits, doc_hits)
    
    # 使用正确的 ChatCompletion API
    resp = client.chat.completions.create(
        model=model, 
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.2
    )
    
    text = resp.choices[0].message.content
    print(f"Raw model response: {text[:2000]}...")  # 调试输出
    
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        # simple last-resort fix: try to extract JSON via braces
        m = re.search(r"\{[\s\S]*\}$", text)
        if not m:
            raise RuntimeError(f"Model did not return JSON. Raw: {text[:300]}…")
        data = json.loads(m.group(0))
    
    # 检查是否返回了 schema 而不是实际数据
    if '$schema' in data and 'type' in data and data.get('type') == 'object':
        # 模型返回了 schema，需要重新提取实际数据
        print("Model returned schema instead of data, extracting actual content...")
        # 查找实际的数据部分
        if 'doc_id' in data and isinstance(data.get('doc_id'), str):
            # 这可能是嵌套的结构，提取实际数据
            actual_data = {}
            for key in ['doc_id', 'primary_type', 'secondary_types', 'dimensions', 'whack_a_mole_flags', 'reasoning_brief', 'confidence', 'disagreements', 'citations']:
                if key in data and key != 'properties':
                    actual_data[key] = data[key]
            data = actual_data
        else:
            # 如果无法提取，返回默认结构
            data = {
                "doc_id": doc_id,
                "primary_type": "undetermined",
                "dimensions": {
                    "utility_dominates": {"value": False, "evidence_spans": []},
                    "on_the_ground": {"value": False, "evidence_spans": []}
                },
                "reasoning_brief": "Unable to process - model returned schema instead of analysis",
                "whack_a_mole_flags": [],
                "secondary_types": [],
                "confidence": 0.0,
                "disagreements": [],
                "citations": []
            }
    
    # 确保 doc_id 存在（如果模型没有返回）
    if 'doc_id' not in data:
        data['doc_id'] = doc_id
    
    # 确保必需字段存在
    if 'primary_type' not in data:
        data['primary_type'] = 'undetermined'
    if 'dimensions' not in data:
        data['dimensions'] = {
            "utility_dominates": {"value": False, "evidence_spans": []},
            "on_the_ground": {"value": False, "evidence_spans": []}
        }
    if 'reasoning_brief' not in data:
        data['reasoning_brief'] = 'Analysis incomplete'
    
    print(f"Final data keys: {list(data.keys())}")  # 调试输出
    
    # Validate schema
    Draft202012Validator(OUTPUT_SCHEMA).validate(data)
    usage = getattr(resp, 'usage', None)
    return {'json': data, 'usage': getattr(usage, 'dict', lambda: None)() if usage else None, 'raw': text}

result = rag_answer(doc_id=f'doc-{RUN_ID}', question='Classify this document under the 4PT framework and answer Q1–Q4 with evidence.', model=LLM_MODEL)
print('OK: received JSON keys:', list(result['json'].keys()))

Raw model response: {
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "type": "object",
  "properties": {
    "doc_id": "doc-20250919-194310",
    "primary_type": "T4",
    "secondary_types": [],
    "dimensions": {
      "utility_dominates": {
        "value": false,
        "evidence_spans": [
          {
            "quote": "the analysis, conclusions, and theories treat individuals, organizations and states as largely self-interested",
            "page_or_loc": "p.40"
          }
        ]
      },
      "on_the_ground": {
        "value": true,
        "evidence_spans": [
          {
            "quote": "the article is either Type 2 or Type 3",
            "page_or_loc": "p.40"
          }
        ]
      }
    },
    "whack_a_mole_flags": [],
    "reasoning_brief": "The analysis does not focus on utility maximization but is grounded in real-world issues.",
    "confidence": 0.8,
    "disagreements": [],
    "citations": []
  }
}...
Model returned schema instead o

In [15]:
# Save Results to JSON
with open(f'outputs/{result["json"]["doc_id"]}.json', 'w', encoding='utf-8') as f:
    json.dump(result['json'], f, ensure_ascii=False, indent=2)
print('Saved to:', f'outputs/{result["json"]["doc_id"]}.json')

Saved to: outputs/doc-20250919-194310.json


In [20]:
def display_analysis_results(result_data: Dict[str, Any]):
    """Display 4PT analysis results in a well-formatted manner"""
    data = result_data['json']
    
    print("=" * 80)
    print(f"📄 4PT Framework Analysis Report")
    print("=" * 80)
    print(f"📌 Document ID: {data['doc_id']}")
    print(f"🎯 Primary Type: {data['primary_type']}")
    if data.get('secondary_types'):
        print(f"🔗 Secondary Types: {', '.join(data['secondary_types'])}")
    print(f"📊 Confidence: {data.get('confidence', 'N/A')}")
    print()
    
    # Display dimension analysis
    print("📋 Dimension Analysis")
    print("-" * 50)
    dims = data.get('dimensions', {})
    
    # Q1: Utility Dominates
    util_dom = dims.get('utility_dominates', {})
    print(f"❓ Q1: Does the reasoning center on utility maximization/satisficing?")
    print(f"   ✅ Answer: {'Yes' if util_dom.get('value') else 'No'}")
    if util_dom.get('evidence_spans'):
        print(f"   📝 Evidence:")
        for i, span in enumerate(util_dom['evidence_spans'][:3], 1):  # Show first 3 items
            quote = span.get('quote', '')[:150] + ('...' if len(span.get('quote', '')) > 150 else '')
            print(f"      {i}. \"{quote}\" ({span.get('page_or_loc', 'N/A')})")
    print()
    
    # Q2: On the Ground
    on_ground = dims.get('on_the_ground', {})
    print(f"❓ Q2: Is the analysis grounded in concrete, on-the-ground problem context?")
    print(f"   ✅ Answer: {'Yes' if on_ground.get('value') else 'No'}")
    if on_ground.get('evidence_spans'):
        print(f"   📝 Evidence:")
        for i, span in enumerate(on_ground['evidence_spans'][:3], 1):
            quote = span.get('quote', '')[:150] + ('...' if len(span.get('quote', '')) > 150 else '')
            print(f"      {i}. \"{quote}\" ({span.get('page_or_loc', 'N/A')})")
    print()
    
    # Q3: Reasoning Brief
    print(f"❓ Q3: Type Classification Reasoning")
    print(f"   💭 {data.get('reasoning_brief', 'N/A')}")
    print()
    
    # Q4: Whack-a-mole Risk
    wam_flags = data.get('whack_a_mole_flags', [])
    print(f"❓ Q4: Any 'whack-a-mole' risk detected?")
    if wam_flags:
        print(f"   ⚠️  Found {len(wam_flags)} potential risk(s):")
        for i, flag in enumerate(wam_flags[:3], 1):
            quote = flag.get('quote', '')[:150] + ('...' if len(flag.get('quote', '')) > 150 else '')
            print(f"      {i}. \"{quote}\" ({flag.get('page_or_loc', 'N/A')})")
    else:
        print(f"   ✅ No obvious risks detected")
    print()
    
    # Citations
    citations = data.get('citations', [])
    if citations:
        print(f"📚 Citations ({len(citations)} items)")
        print("-" * 30)
        for i, cite in enumerate(citations[:5], 1):  # Show first 5 citations
            print(f"   {i}. {cite.get('source', 'N/A')} - {cite.get('page', 'N/A')}")
        if len(citations) > 5:
            print(f"   ... and {len(citations) - 5} more citations")
    
    print("=" * 80)
    print("✨ Analysis Complete!")
    print("=" * 80)

# Display formatted analysis results
print("\n🎨 Here are the formatted analysis results:\n")
display_analysis_results(result)



🎨 Here are the formatted analysis results:

📄 4PT Framework Analysis Report
📌 Document ID: doc-20250919-194310
🎯 Primary Type: undetermined
📊 Confidence: 0.0

📋 Dimension Analysis
--------------------------------------------------
❓ Q1: Does the reasoning center on utility maximization/satisficing?
   ✅ Answer: No

❓ Q2: Is the analysis grounded in concrete, on-the-ground problem context?
   ✅ Answer: No

❓ Q3: Type Classification Reasoning
   💭 Unable to process - model returned schema instead of analysis

❓ Q4: Any 'whack-a-mole' risk detected?
   ✅ No obvious risks detected

✨ Analysis Complete!
